In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [3]:
file_path = r"C:\Users\priva\OneDrive\Documents\Retail\Online Retail.csv"
df = pd.read_csv(file_path, encoding="ISO-8859-1")

In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes)

Shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Dtypes:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


In [5]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,"17,850.00",United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,"17,850.00",United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,"17,850.00",United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,"17,850.00",United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,"17,850.00",United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,"13,047.00",United Kingdom


In [6]:
df.isna().sum().sort_values(ascending=False)

CustomerID     135080
Description      1454
InvoiceNo           0
StockCode           0
Quantity            0
InvoiceDate         0
UnitPrice           0
Country             0
dtype: int64

In [7]:
# basic date conversion
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# useful flags
df["is_cancel_invoice"] = df["InvoiceNo"].astype(str).str.startswith("C")
df["is_negative_qty"] = df["Quantity"] < 0
df["is_nonpositive_price"] = df["UnitPrice"] <= 0
df["missing_customer"] = df["CustomerID"].isna()

# line-level amount
df["line_revenue"] = df["Quantity"] * df["UnitPrice"]

In [8]:
summary = pd.Series({
    "rows": len(df),
    "unique_invoices": df["InvoiceNo"].nunique(),
    "unique_customers_incl_na": df["CustomerID"].nunique(dropna=True),
    "cancel_invoice_rows": df["is_cancel_invoice"].sum(),
    "negative_qty_rows": df["is_negative_qty"].sum(),
    "nonpositive_price_rows": df["is_nonpositive_price"].sum(),
    "missing_customer_rows": df["missing_customer"].sum()
})

summary

rows                        541909
unique_invoices              25900
unique_customers_incl_na      4372
cancel_invoice_rows           9288
negative_qty_rows            10624
nonpositive_price_rows        2517
missing_customer_rows       135080
dtype: int64

In [9]:
# exact duplicate rows across all columns
exact_dupes = df.duplicated().sum()
exact_dupes

np.int64(5268)

In [10]:
# look at some exact duplicates if they exist
df[df.duplicated(keep=False)].sort_values(list(df.columns[:8])).head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancel_invoice,is_negative_qty,is_nonpositive_price,missing_customer,line_revenue
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom,False,False,False,False,1.25
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom,False,False,False,False,1.25
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom,False,False,False,False,4.95
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom,False,False,False,False,4.95
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom,False,False,False,False,2.10
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom,False,False,False,False,2.10
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,"17,908.00",United Kingdom,False,False,False,False,2.95
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,"17,908.00",United Kingdom,False,False,False,False,2.95
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,"17,920.00",United Kingdom,False,False,False,False,1.65
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,"17,920.00",United Kingdom,False,False,False,False,1.65


In [11]:
df_raw = df.copy()

df = df.drop_duplicates().copy()

df["CustomerID"] = df["CustomerID"].astype("Int64")
df["Description"] = df["Description"].str.strip()

print("Raw shape:", df_raw.shape)
print("After exact duplicate removal:", df.shape)
print("Removed rows:", len(df_raw) - len(df))

Raw shape: (541909, 13)
After exact duplicate removal: (536641, 13)
Removed rows: 5268


In [12]:
df["txn_type"] = np.select(
    [
        df["is_cancel_invoice"] | df["is_negative_qty"],
        (~df["is_cancel_invoice"]) & (~df["is_negative_qty"])
    ],
    [
        "return_or_cancel",
        "purchase"
    ],
    default="other"
)

df["gross_revenue"] = np.where(
    df["txn_type"] == "purchase",
    df["Quantity"] * df["UnitPrice"],
    0
)

df["net_revenue"] = df["Quantity"] * df["UnitPrice"]

In [13]:
df["txn_type"].value_counts(dropna=False)

txn_type
purchase            526054
return_or_cancel     10587
Name: count, dtype: int64

In [14]:
pd.Series({
    "rows_after_dupes": len(df),
    "known_customer_rows": df["CustomerID"].notna().sum(),
    "missing_customer_rows": df["CustomerID"].isna().sum(),
    "purchase_rows": (df["txn_type"] == "purchase").sum(),
    "return_rows": (df["txn_type"] == "return_or_cancel").sum(),
    "nonpositive_price_rows": (df["UnitPrice"] <= 0).sum()
})

rows_after_dupes          536641
known_customer_rows       401604
missing_customer_rows     135037
purchase_rows             526054
return_rows                10587
nonpositive_price_rows      2512
dtype: int64

In [15]:
df[df["UnitPrice"] <= 0].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancel_invoice,is_negative_qty,is_nonpositive_price,missing_customer,line_revenue,txn_type,gross_revenue,net_revenue
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.00,<NA>,United Kingdom,False,False,True,True,0.00,purchase,0.00,0.00
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.00,<NA>,United Kingdom,False,True,True,True,-0.00,return_or_cancel,0.00,-0.00


In [16]:
df_customer = df[df["CustomerID"].notna()].copy()

pd.Series({
    "customer_rows": len(df_customer),
    "unique_customers": df_customer["CustomerID"].nunique(),
    "unique_invoices": df_customer["InvoiceNo"].nunique()
})

customer_rows       401604
unique_customers      4372
unique_invoices      22190
dtype: int64

In [17]:
df_customer[[
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country",
    "txn_type", "gross_revenue", "net_revenue"
]].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,txn_type,gross_revenue,net_revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,purchase,15.30,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,purchase,20.34,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,purchase,22.00,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,purchase,20.34,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,purchase,20.34,20.34
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,purchase,15.30,15.30
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,purchase,25.50,25.50
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,purchase,11.10,11.10
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,purchase,11.10,11.10
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,purchase,54.08,54.08


In [18]:
df_customer = df[
    (df["CustomerID"].notna()) &
    (df["UnitPrice"] > 0)
].copy()

print("Shape:", df_customer.shape)
print("Unique customers:", df_customer["CustomerID"].nunique())
print("Unique invoices:", df_customer["InvoiceNo"].nunique())

Shape: (401564, 16)
Unique customers: 4371
Unique invoices: 22186


In [19]:
pd.Series({
    "purchase_rows": (df_customer["txn_type"] == "purchase").sum(),
    "return_rows": (df_customer["txn_type"] == "return_or_cancel").sum(),
    "total_rows": len(df_customer)
})

purchase_rows    392692
return_rows        8872
total_rows       401564
dtype: int64

In [20]:
df_customer[[
    "InvoiceNo", "Quantity", "UnitPrice",
    "txn_type", "gross_revenue", "net_revenue"
]].describe()

,Quantity,UnitPrice,gross_revenue,net_revenue
count,"401,564.00","401,564.00","401,564.00","401,564.00"
mean,12.15,3.47,22.13,20.62
std,249.51,69.77,307.66,430.37
min,"-80,995.00",0.00,0.00,"-168,469.60"
25%,2.00,1.25,4.25,4.25
50%,5.00,1.95,11.70,11.70
75%,12.00,3.75,19.80,19.80
max,"80,995.00","38,970.00","168,469.60","168,469.60"


In [21]:
customer_df = df_customer.groupby("CustomerID").agg(
    total_orders=("InvoiceNo", "nunique"),
    total_items=("Quantity", "sum"),
    gross_revenue=("gross_revenue", "sum"),
    net_revenue=("net_revenue", "sum"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

In [22]:
# recency (days since last purchase)
snapshot_date = df_customer["InvoiceDate"].max() + pd.Timedelta(days=1)

customer_df["recency_days"] = (
    snapshot_date - customer_df["last_purchase"]
).dt.days

In [23]:

customer_df.describe()

,CustomerID,total_orders,total_items,gross_revenue,net_revenue,first_purchase,last_purchase,recency_days
count,"4,371.00","4,371.00","4,371.00","4,371.00","4,371.00",4371,4371,"4,371.00"
mean,"15,300.15",5.08,"1,116.21","2,033.22","1,893.96",2011-04-28 04:36:58.352779520,2011-09-08 22:46:30.871654144,92.06
min,"12,346.00",1.00,-303.00,0.00,"-4,287.63",2010-12-01 08:26:00,2010-12-01 09:53:00,1.00
25%,"13,813.50",1.00,151.50,302.23,291.94,2011-01-12 12:43:00,2011-07-19 15:13:00,17.00
50%,"15,301.00",3.00,364.00,659.46,644.24,2011-03-31 17:59:00,2011-10-20 15:56:00,50.00
75%,"16,778.50",5.00,956.00,"1,647.37","1,608.94",2011-08-16 12:28:00,2011-11-23 09:25:00,143.00
max,"18,287.00",248.00,"196,143.00","280,206.02","279,489.02",2011-12-09 12:16:00,2011-12-09 12:50:00,374.00
std,"1,722.31",9.33,"4,662.70","8,953.00","8,219.59",NaN,NaN,100.77


In [24]:
customer_df.sort_values("gross_revenue", ascending=False).head(10)

,CustomerID,total_orders,total_items,gross_revenue,net_revenue,first_purchase,last_purchase,recency_days
1702,14646,76,196143,"280,206.02","279,489.02",2010-12-20 10:09:00,2011-12-08 12:12:00,2
4232,18102,62,64122,"259,657.30","256,438.49",2010-12-07 16:42:00,2011-12-09 11:50:00,1
3757,17450,55,69009,"194,390.79","187,322.17",2010-12-07 09:23:00,2011-12-01 13:29:00,8
3032,16446,3,2,"168,472.50",2.90,2011-05-18 09:52:00,2011-12-09 09:27:00,1
1894,14911,248,76905,"143,711.17","132,458.73",2010-12-01 14:05:00,2011-12-08 15:54:00,1
55,12415,26,76946,"124,914.53","123,725.45",2011-01-06 11:12:00,2011-11-15 14:22:00,24
1344,14156,66,56908,"117,210.08","113,214.59",2010-12-02 17:08:00,2011-11-30 10:54:00,10
3800,17511,46,63012,"91,062.38","88,125.38",2010-12-01 10:19:00,2011-12-07 10:12:00,3
2721,16029,76,33632,"80,850.84","53,168.69",2010-12-01 09:57:00,2011-11-01 10:27:00,39
0,12346,2,0,"77,183.60",0.00,2011-01-18 10:01:00,2011-01-18 10:17:00,326


In [25]:
# total revenue
total_revenue = customer_df["net_revenue"].sum()

# sort customers by value
customer_df_sorted = customer_df.sort_values("net_revenue", ascending=False)

# cumulative revenue
customer_df_sorted["cum_revenue"] = customer_df_sorted["net_revenue"].cumsum()
customer_df_sorted["cum_pct"] = customer_df_sorted["cum_revenue"] / total_revenue

customer_df_sorted.head(10)

,CustomerID,total_orders,total_items,gross_revenue,net_revenue,first_purchase,last_purchase,recency_days,cum_revenue,cum_pct
1702,14646,76,196143,"280,206.02","279,489.02",2010-12-20 10:09:00,2011-12-08 12:12:00,2,"279,489.02",0.03
4232,18102,62,64122,"259,657.30","256,438.49",2010-12-07 16:42:00,2011-12-09 11:50:00,1,"535,927.51",0.06
3757,17450,55,69009,"194,390.79","187,322.17",2010-12-07 09:23:00,2011-12-01 13:29:00,8,"723,249.68",0.09
1894,14911,248,76905,"143,711.17","132,458.73",2010-12-01 14:05:00,2011-12-08 15:54:00,1,"855,708.41",0.10
55,12415,26,76946,"124,914.53","123,725.45",2011-01-06 11:12:00,2011-11-15 14:22:00,24,"979,433.86",0.12
1344,14156,66,56908,"117,210.08","113,214.59",2010-12-02 17:08:00,2011-11-30 10:54:00,10,"1,092,648.45",0.13
3800,17511,46,63012,"91,062.38","88,125.38",2010-12-01 10:19:00,2011-12-07 10:12:00,3,"1,180,773.83",0.14
3201,16684,31,49390,"66,653.56","65,892.08",2010-12-16 17:34:00,2011-12-05 14:06:00,4,"1,246,665.91",0.15
1004,13694,60,61899,"65,039.62","62,690.54",2010-12-01 12:12:00,2011-12-06 09:32:00,4,"1,309,356.45",0.16
2191,15311,118,37673,"60,632.75","59,284.19",2010-12-01 09:41:00,2011-12-09 12:00:00,1,"1,368,640.64",0.17


In [26]:
# how many customers generate 50%, 80%, 90% of revenue
def find_cutoff(pct):
    return (customer_df_sorted["cum_pct"] <= pct).sum()

pd.Series({
    "top_customers_for_50%": find_cutoff(0.5),
    "top_customers_for_80%": find_cutoff(0.8),
    "top_customers_for_90%": find_cutoff(0.9),
    "total_customers": len(customer_df_sorted)
})

top_customers_for_50%     238
top_customers_for_80%    1166
top_customers_for_90%    1929
total_customers          4371
dtype: int64

In [27]:
# revenue percentiles
customer_df["net_revenue"].quantile([0.5, 0.75, 0.9, 0.95, 0.99])

0.50      644.24
0.75    1,608.94
0.90    3,497.14
0.95    5,601.45
0.99   17,231.39
Name: net_revenue, dtype: float64

In [28]:
p50 = customer_df["net_revenue"].quantile(0.5)
p90 = customer_df["net_revenue"].quantile(0.9)

def value_segment(x):
    if x >= p90:
        return "high"
    elif x >= p50:
        return "mid"
    else:
        return "low"

customer_df["value_segment"] = customer_df["net_revenue"].apply(value_segment)

In [29]:
customer_df["value_segment"].value_counts()

value_segment
low     2185
mid     1748
high     438
Name: count, dtype: int64

In [30]:
customer_df.groupby("value_segment")["net_revenue"].agg(["count", "mean", "sum"])

,count,mean,sum
value_segment,,,
high,438,"11,366.67","4,978,602.83"
low,2185,296.63,"648,131.37"
mid,1748,"1,517.04","2,651,785.22"


In [31]:
# frequency tiers (orders)
customer_df["freq_segment"] = pd.qcut(
    customer_df["total_orders"],
    q=3,
    labels=["low_freq", "mid_freq", "high_freq"]
)

In [32]:
# recency tiers (lower = better)
customer_df["recency_segment"] = pd.qcut(
    customer_df["recency_days"],
    q=3,
    labels=["recent", "mid_recency", "old"]
)

In [33]:
customer_df[["value_segment", "freq_segment", "recency_segment"]].head()

,value_segment,freq_segment,recency_segment
0,low,low_freq,old
1,high,high_freq,recent
2,mid,mid_freq,mid_recency
3,mid,low_freq,recent
4,low,low_freq,old


In [34]:
customer_df["behavior_segment"] = (
    customer_df["value_segment"] + "_" +
    customer_df["freq_segment"].astype(str) + "_" +
    customer_df["recency_segment"].astype(str)
)

In [35]:
customer_df["behavior_segment"].value_counts().head(10)

behavior_segment
low_low_freq_old             958
low_low_freq_mid_recency     542
mid_high_freq_recent         481
high_high_freq_recent        332
mid_high_freq_mid_recency    328
low_low_freq_recent          284
mid_mid_freq_mid_recency     211
mid_mid_freq_recent          190
mid_low_freq_mid_recency     148
mid_low_freq_old             132
Name: count, dtype: int64

In [36]:
segment_summary = customer_df.groupby("behavior_segment").agg(
    customers=("CustomerID", "count"),
    total_revenue=("net_revenue", "sum"),
    avg_revenue=("net_revenue", "mean")
).sort_values("total_revenue", ascending=False)

segment_summary.head(10)

,customers,total_revenue,avg_revenue
behavior_segment,,,
high_high_freq_recent,332,"4,272,816.30","12,869.93"
mid_high_freq_recent,481,"905,229.31","1,881.97"
mid_high_freq_mid_recency,328,"609,352.89","1,857.78"
high_high_freq_mid_recency,66,"479,114.83","7,259.32"
mid_mid_freq_mid_recency,211,"268,335.72","1,271.73"
low_low_freq_old,958,"234,467.26",244.75
mid_mid_freq_recent,190,"229,083.68","1,205.70"
low_low_freq_mid_recency,542,"162,456.31",299.73
mid_low_freq_mid_recency,148,"152,513.50","1,030.50"


In [37]:
# compress into main 5 segments
def simplify_segment(row):
    if row["value_segment"] == "high" and row["freq_segment"] == "high_freq" and row["recency_segment"] == "recent":
        return "champions"
    elif row["value_segment"] == "mid" and row["freq_segment"] == "high_freq" and row["recency_segment"] == "recent":
        return "potential_loyalists"
    elif row["recency_segment"] == "old":
        return "at_risk_or_lost"
    elif row["value_segment"] == "low":
        return "low_value"
    else:
        return "others"

customer_df["main_segment"] = customer_df.apply(simplify_segment, axis=1)

In [38]:
customer_df.groupby("main_segment")["net_revenue"].agg(["count", "sum", "mean"])

,count,sum,mean
main_segment,,,
at_risk_or_lost,1454,"834,111.19",573.67
champions,332,"4,272,816.30","12,869.93"
low_value,1083,"358,849.70",331.35
others,1021,"1,907,512.92","1,868.28"
potential_loyalists,481,"905,229.31","1,881.97"


In [39]:
segment_kpis = customer_df.groupby("main_segment").agg(
    customers=("CustomerID", "count"),
    revenue=("net_revenue", "sum")
).reset_index()

segment_kpis["customer_pct"] = segment_kpis["customers"] / segment_kpis["customers"].sum()
segment_kpis["revenue_pct"] = segment_kpis["revenue"] / segment_kpis["revenue"].sum()
segment_kpis["avg_revenue_per_customer"] = segment_kpis["revenue"] / segment_kpis["customers"]

segment_kpis.sort_values("revenue", ascending=False)

,main_segment,customers,revenue,customer_pct,revenue_pct,avg_revenue_per_customer
1,champions,332,"4,272,816.30",0.08,0.52,"12,869.93"
3,others,1021,"1,907,512.92",0.23,0.23,"1,868.28"
4,potential_loyalists,481,"905,229.31",0.11,0.11,"1,881.97"
0,at_risk_or_lost,1454,"834,111.19",0.33,0.10,573.67
2,low_value,1083,"358,849.70",0.25,0.04,331.35


In [40]:
segment_kpis["revenue_to_customer_ratio"] = (
    segment_kpis["revenue_pct"] / segment_kpis["customer_pct"]
)

segment_kpis.sort_values("revenue_to_customer_ratio", ascending=False)

,main_segment,customers,revenue,customer_pct,revenue_pct,avg_revenue_per_customer,revenue_to_customer_ratio
1,champions,332,"4,272,816.30",0.08,0.52,"12,869.93",6.80
4,potential_loyalists,481,"905,229.31",0.11,0.11,"1,881.97",0.99
3,others,1021,"1,907,512.92",0.23,0.23,"1,868.28",0.99
0,at_risk_or_lost,1454,"834,111.19",0.33,0.10,573.67,0.30
2,low_value,1083,"358,849.70",0.25,0.04,331.35,0.17


In [41]:
customer_df.groupby("main_segment")[["total_orders", "recency_days", "net_revenue"]].mean().sort_values("net_revenue", ascending=False)

,total_orders,recency_days,net_revenue
main_segment,,,
champions,23.45,8.33,"12,869.93"
potential_loyalists,8.46,9.98,"1,881.97"
others,5.00,40.41,"1,868.28"
at_risk_or_lost,2.13,214.65,573.67
low_value,1.97,38.31,331.35


In [42]:
# average order value (AOV)
order_df = df_customer.groupby(["InvoiceNo", "CustomerID"]).agg(
    order_revenue=("net_revenue", "sum"),
    total_items=("Quantity", "sum"),
    order_date=("InvoiceDate", "min")
).reset_index()

# merge segment info
order_df = order_df.merge(
    customer_df[["CustomerID", "main_segment"]],
    on="CustomerID",
    how="left"
)

In [43]:
order_df.groupby("main_segment").agg(
    avg_order_value=("order_revenue", "mean"),
    avg_items_per_order=("total_items", "mean"),
    total_orders=("InvoiceNo", "count")
).sort_values("avg_order_value", ascending=False)

,avg_order_value,avg_items_per_order,total_orders
main_segment,,,
champions,548.92,312.06,7784
others,373.66,227.13,5105
at_risk_or_lost,269.33,162.85,3097
potential_loyalists,222.47,134.60,4069
low_value,168.39,111.83,2131


In [44]:
# purchase frequency proxy (orders per customer)
customer_df.groupby("main_segment")["total_orders"].mean().sort_values(ascending=False)

main_segment
champions             23.45
potential_loyalists    8.46
others                 5.00
at_risk_or_lost        2.13
low_value              1.97
Name: total_orders, dtype: float64

In [45]:
# number of unique products per customer
product_diversity = df_customer.groupby("CustomerID").agg(
    unique_products=("StockCode", "nunique")
).reset_index()

customer_df = customer_df.merge(product_diversity, on="CustomerID", how="left")

In [46]:
customer_df.groupby("main_segment")["unique_products"].mean().sort_values(ascending=False)

main_segment
champions             200.06
potential_loyalists    99.93
others                 78.49
low_value              29.05
at_risk_or_lost        28.56
Name: unique_products, dtype: float64

In [47]:
customer_df.groupby("main_segment")["unique_products"].describe()

,count,mean,std,min,25%,50%,75%,max
main_segment,,,,,,,,
at_risk_or_lost,"1,454.00",28.56,31.88,1.00,9.00,19.00,36.00,421.00
champions,332.00,200.06,201.00,8.00,92.75,156.50,244.50,"1,794.00"
low_value,"1,083.00",29.05,27.10,1.00,11.00,22.00,36.00,197.00
others,"1,021.00",78.49,58.21,1.00,39.00,61.00,104.00,406.00
potential_loyalists,481.00,99.93,74.92,3.00,50.00,82.00,123.00,580.00


In [48]:
customer_df.groupby("main_segment").agg(
    avg_items_per_order=("total_items", lambda x: x.sum() / customer_df.loc[x.index, "total_orders"].sum()),
    avg_unique_products=("unique_products", "mean"),
    avg_orders=("total_orders", "mean")
).sort_values("avg_items_per_order", ascending=False)

,avg_items_per_order,avg_unique_products,avg_orders
main_segment,,,
champions,312.06,200.06,23.45
others,227.13,78.49,5.00
at_risk_or_lost,162.85,28.56,2.13
potential_loyalists,134.60,99.93,8.46
low_value,111.83,29.05,1.97


In [49]:
# ratio: unique products per order
customer_df["product_per_order"] = (
    customer_df["unique_products"] / customer_df["total_orders"]
)

customer_df.groupby("main_segment")["product_per_order"].mean().sort_values(ascending=False)

main_segment
others                20.29
low_value             17.64
at_risk_or_lost       15.79
potential_loyalists   12.56
champions             11.85
Name: product_per_order, dtype: float64

In [50]:
# purchase-only customer-product quantity
cust_prod = df_customer[df_customer["txn_type"] == "purchase"].groupby(
    ["CustomerID", "StockCode"]
).agg(
    total_qty=("Quantity", "sum")
).reset_index()

In [51]:
# total purchased quantity per customer
cust_total = cust_prod.groupby("CustomerID")["total_qty"].sum().reset_index()
cust_total.columns = ["CustomerID", "customer_total_qty"]

cust_prod = cust_prod.merge(cust_total, on="CustomerID", how="left")

In [52]:
# product share within customer purchases
cust_prod["product_share"] = cust_prod["total_qty"] / cust_prod["customer_total_qty"]

In [53]:
# top product share per customer
top_product_share = cust_prod.groupby("CustomerID")["product_share"].max().reset_index()
top_product_share.columns = ["CustomerID", "top_product_share"]

customer_df = customer_df.drop(columns=["top_product_share"], errors="ignore")
customer_df = customer_df.merge(top_product_share, on="CustomerID", how="left")

In [54]:
customer_df.groupby("main_segment")["top_product_share"].mean().sort_values(ascending=False)

main_segment
at_risk_or_lost       0.25
low_value             0.23
others                0.12
potential_loyalists   0.11
champions             0.10
Name: top_product_share, dtype: float64

In [55]:
customer_df.groupby("main_segment")["top_product_share"].describe()

,count,mean,std,min,25%,50%,75%,max
main_segment,,,,,,,,
at_risk_or_lost,"1,424.00",0.25,0.24,0.02,0.10,0.17,0.30,1.00
champions,332.00,0.10,0.11,0.01,0.04,0.06,0.12,0.95
low_value,"1,080.00",0.23,0.19,0.03,0.11,0.16,0.28,1.00
others,"1,021.00",0.12,0.13,0.01,0.05,0.08,0.14,1.00
potential_loyalists,481.00,0.11,0.11,0.02,0.05,0.08,0.13,0.71


In [56]:
# first purchase per customer
first_purchase = df_customer.groupby("CustomerID")["InvoiceDate"].min().reset_index()
first_purchase.columns = ["CustomerID", "first_purchase_date"]

df_customer = df_customer.merge(first_purchase, on="CustomerID", how="left")

In [57]:
# flag first vs repeat transactions
df_customer["is_first_purchase"] = (
    df_customer["InvoiceDate"] == df_customer["first_purchase_date"]
)

In [58]:
# orders per customer
orders_per_customer = df_customer.groupby("CustomerID")["InvoiceNo"].nunique()

# classify customers
customer_df["is_repeat_customer"] = customer_df["total_orders"] > 1

In [59]:
customer_df["is_repeat_customer"].value_counts(normalize=True)

is_repeat_customer
True    0.70
False   0.30
Name: proportion, dtype: float64

In [60]:
# revenue split: repeat vs one-time
customer_df.groupby("is_repeat_customer")["net_revenue"].agg(["count", "sum", "mean"])

,count,sum,mean
is_repeat_customer,,,
False,1312,"431,608.86",328.97
True,3059,"7,846,910.56","2,565.19"


In [61]:
order_counts = customer_df["total_orders"].copy()

thresholds = [1, 2, 3, 4, 5, 10, 25]

progression = []
for i in range(len(thresholds) - 1):
    a = thresholds[i]
    b = thresholds[i + 1]
    
    customers_at_a = (order_counts >= a).sum()
    customers_at_b = (order_counts >= b).sum()
    
    progression.append({
        "from_orders": a,
        "to_orders": b,
        "customers_at_least_from": customers_at_a,
        "customers_at_least_to": customers_at_b,
        "progression_rate": customers_at_b / customers_at_a if customers_at_a > 0 else np.nan
    })

progression_df = pd.DataFrame(progression)
progression_df

,from_orders,to_orders,customers_at_least_from,customers_at_least_to,progression_rate
0,1,2,4371,3059,0.70
1,2,3,3059,2242,0.73
2,3,4,2242,1752,0.78
3,4,5,1752,1374,0.78
4,5,10,1374,537,0.39
5,10,25,537,104,0.19


In [62]:
# full progression from 1 to 25
full_progression = []
for n in range(1, 25):
    customers_at_n = (order_counts >= n).sum()
    customers_at_next = (order_counts >= (n + 1)).sum()
    
    full_progression.append({
        "from_orders": n,
        "to_orders": n + 1,
        "customers_at_least_from": customers_at_n,
        "customers_at_least_to": customers_at_next,
        "progression_rate": customers_at_next / customers_at_n if customers_at_n > 0 else np.nan
    })

full_progression_df = pd.DataFrame(full_progression)
full_progression_df.head(10)

,from_orders,to_orders,customers_at_least_from,customers_at_least_to,progression_rate
0,1,2,4371,3059,0.70
1,2,3,3059,2242,0.73
2,3,4,2242,1752,0.78
3,4,5,1752,1374,0.78
4,5,6,1374,1087,0.79
5,6,7,1087,891,0.82
6,7,8,891,734,0.82
7,8,9,734,617,0.84
8,9,10,617,537,0.87
9,10,11,537,459,0.85


In [63]:
full_progression_df.tail(10)

,from_orders,to_orders,customers_at_least_from,customers_at_least_to,progression_rate
14,15,16,267,239,0.90
15,16,17,239,213,0.89
16,17,18,213,194,0.91
17,18,19,194,172,0.89
18,19,20,172,156,0.91
19,20,21,156,142,0.91
20,21,22,142,132,0.93
21,22,23,132,119,0.90
22,23,24,119,110,0.92
23,24,25,110,104,0.95


In [64]:
customer_orders = df_customer.groupby(["CustomerID", "InvoiceNo"]).agg(
    order_date=("InvoiceDate", "min")
).reset_index()

customer_orders = customer_orders.sort_values(["CustomerID", "order_date"])
customer_orders["order_number"] = customer_orders.groupby("CustomerID").cumcount() + 1

In [65]:
first_orders = customer_orders[customer_orders["order_number"] == 1][
    ["CustomerID", "order_date"]
].rename(columns={"order_date": "first_order_date"})

second_orders = customer_orders[customer_orders["order_number"] == 2][
    ["CustomerID", "order_date"]
].rename(columns={"order_date": "second_order_date"})

In [66]:
time_to_second = first_orders.merge(second_orders, on="CustomerID", how="inner")
time_to_second["days_to_second_order"] = (
    time_to_second["second_order_date"] - time_to_second["first_order_date"]
).dt.days

time_to_second.head()

,CustomerID,first_order_date,second_order_date,days_to_second_order
0,12346,2011-01-18 10:01:00,2011-01-18 10:17:00,0
1,12347,2010-12-07 14:57:00,2011-01-26 14:30:00,49
2,12348,2010-12-16 19:09:00,2011-01-25 10:42:00,39
3,12352,2011-02-16 12:33:00,2011-03-01 14:57:00,13
4,12356,2011-01-18 09:50:00,2011-04-08 12:33:00,80


In [67]:
time_to_second["days_to_second_order"].describe()

count   3,059.00
mean       62.46
std        74.98
min         0.00
25%         7.00
50%        33.00
75%        94.00
max       365.00
Name: days_to_second_order, dtype: float64

In [68]:
time_to_second["days_to_second_order"].quantile([0.25, 0.5, 0.75, 0.9])

0.25     7.00
0.50    33.00
0.75    94.00
0.90   169.00
Name: days_to_second_order, dtype: float64